# Relational Reasoning Benchmark — Report

Loads `results/summary.csv` and `results/raw/<prompt_variant>/` and renders per-variant × per-model accuracy (broken out by prompt variant: `cot` vs `no_cot`), scale-test curve, error-type breakdown, and per-problem drill-down. Rerun all cells after each benchmark run.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
RESULTS = REPO / "results"
RAW = RESULTS / "raw"

summary = pd.read_csv(RESULTS / "summary.csv")
summary.head()

## Accuracy by variant and model

In [ ]:
pivot = (
    summary.groupby(["prompt_variant", "variant", "model"], as_index=False)
    .agg(accuracy=("accuracy", "mean"),
         se=("accuracy", lambda x: x.std(ddof=1) / (len(x) ** 0.5)),
         n_instances=("problem_id", "nunique"))
)
display_pivot = (
    pivot.pivot(index=["prompt_variant", "variant"], columns="model", values="accuracy")
    .round(3)
)
display_pivot

## Scale-test curve

In [ ]:
scale_rows = summary[summary["variant"] == "scale"].copy()
# Extract n_objects from the problem JSON.
def _n_objects(problem_id):
    path = REPO / "problems" / f"{problem_id}.json"
    return json.loads(path.read_text())["metadata"]["n_objects"]

scale_rows["n_objects"] = scale_rows["problem_id"].map(_n_objects)

# Solid = cot, dashed = no_cot.
linestyle_for = {"cot": "-", "no_cot": "--"}

fig, ax = plt.subplots(figsize=(7, 4))
for (model, pv), g in scale_rows.groupby(["model", "prompt_variant"]):
    g = g.sort_values("n_objects")
    ax.plot(
        g["n_objects"], g["accuracy"],
        marker="o", linestyle=linestyle_for.get(pv, "-"),
        label=f"{model} ({pv})",
    )
ax.set_xlabel("Number of objects")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("Scale-test accuracy vs. problem size")
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
fig.tight_layout()
plt.show()

## Error-type breakdown

In [ ]:
error_cols = ["n_correct", "n_feature_match", "n_positional_match", "n_other", "n_parse_error"]
by_group = summary.groupby(["prompt_variant", "model"])[error_cols].sum()
# Normalize to proportions of samples.
by_group_pct = by_group.div(by_group.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(8, 4))
by_group_pct.plot(kind="bar", stacked=True, ax=ax)
ax.set_ylabel("Proportion of samples")
ax.set_title("Outcome and error-type breakdown by (prompt_variant, model)")
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
fig.tight_layout()
plt.show()

## Per-problem drill-down

In [ ]:
drill = summary.sort_values(["variant", "problem_id", "prompt_variant", "model"])
drill[["prompt_variant", "model", "variant", "problem_id", "accuracy",
       "n_feature_match", "n_positional_match", "n_parse_error"]]

## Inspect a single response

Change `prompt_variant`, `model_id`, `problem_id`, and `sample_n` below to view a specific model output.

In [ ]:
prompt_variant = "cot"
model_id = "claude-haiku-4-5-20251001"
problem_id = "baseline_00"
sample_n = 0

path = RAW / prompt_variant / model_id / problem_id / f"sample_{sample_n}.json"
rec = json.loads(path.read_text())
print("=== PROMPT ===\n")
print(rec["prompt"])
print("\n=== RESPONSE ===\n")
print(rec["response_text"])
print("\n=== SCORE ===")
print({k: rec[k] for k in ["is_correct", "error_type", "parsed_answer", "correct_answer"]})